In [17]:
import os
import re
import json
import pandas as pd
from pathlib import Path
from tqdm import tqdm

In [18]:
def normalize_legal_name(name):
    """
    Creates a 'fingerprint' for comparison by removing legal noise.
    """
    if not name: return ""
    n = name.lower()
    # Remove m/s, versus, v., and common suffixes
    n = re.sub(r'\bm/s\.?\b|\bversus\b|\bv\.?s?\.?\b', ' ', n)
    n = re.sub(r'\band\s+(?:ors?|anr\.?)\b', ' ', n)
    # Return only alphanumeric for a 'fuzzy' match
    return re.sub(r'[^a-z0-9]', '', n)

In [19]:
def extract_precedents(text):
    """
    Identifies cited cases while ensuring the full 'v.' pair is captured.
    """
    # Look for [Capitalized Name] v. [Capitalized Name]
    # We allow more characters after 'v.' but stop before typical noise words
    pattern = r"([A-Z][\w\s\.\&]+ v\. [A-Z][\w\s\.\&]+)"
    found = re.findall(pattern, text)
    
    cleaned = []
    for p in found:
        p_clean = p.replace('\n', ' ').strip()
        # Only cut if we hit specific legal document keywords, not just any word
        # This prevents cutting 'Commissioner of Central Excise' into just 'Commissioner'
        p_clean = re.split(r' (?:show cause|held|provisions|EXIM Policy|dated|under Section)', p_clean)[0]
        
        if len(p_clean) > 10 and ' v. ' in p_clean:
            cleaned.append(p_clean.strip())
            
    return list(set(cleaned))

In [20]:
def extract_from_html(html_content):
    """
    Parses professional metadata from SCR HTML.
    """
    data = {
        "case_name": None,
        "coram": None, 
        "decision_date": None, 
        "case_no": None, 
        "disposal_nature": None
    }
    
    
    if not html_content: 
        return data
    
    patterns = {
        "case_name": r"<strong>(.*?)</strong>",
        "coram": r"Coram : (.*?)<br>",
        "decision_date": r"Decision Date :</span><font color='green'> (.*?)</font>",
        "case_no": r"Case No :</span><font color='green'> (.*?)</font>",
        "disposal_nature": r"Disposal Nature :</span><font color='green'> (.*?)</font>"
    }
    
    for key, pattern in patterns.items():
        match = re.search(pattern, html_content)
        if match:
            # Strips residual HTML like <strong> or <sup>
            clean_val = re.sub(r'<.*?>', '', match.group(1)).strip()
            data[key] = clean_val
    return data

In [21]:
# def build_final_parquet(text_folder, metadata_root, output_path):
#     # ... (Keep your existing path setup) ...
#     all_records = []
#     text_files = [f for f in os.listdir(text_folder) if f.endswith('.txt')]
    
#     for filename in tqdm(text_files, desc="Building Dataset"):
#         with open(os.path.join(text_folder, filename), 'r', encoding='utf-8') as f:
#             content = f.read()

#         base_id = filename.replace('_EN.txt', '').replace('.txt', '')
#         year = base_id.split('_')[0]
#         json_path = os.path.join(metadata_root, year, "metadata", f"{base_id}.json")

#         # 1. Initialize record placeholders
#         record = {
#             "file_id": base_id, 
#             "year": year, 
#             "full_text": content,
#             "case_name": None, 
#             "precedents": [], 
#             "coram": None,
#             "decision_date": None, 
#             "case_no": None, 
#             "disposal_nature": None,
#             "neutral_citation": None
#         }

#         # 2. Extract Metadata from JSON
#         if os.path.exists(json_path):
#             with open(json_path, 'r', encoding='utf-8') as jf:
#                 meta_json = json.load(jf)
#                 record.update(extract_from_html(meta_json.get("raw_html", "")))
#                 record["neutral_citation"] = meta_json.get("nc_display")

#         # 3. Apply the Self-Citation Filter
#         raw_precedents = extract_precedents(content)
#         current_name_fingerprint = normalize_legal_name(record["case_name"])
        
#         filtered_precedents = []
#         for p in raw_precedents:
#             # Check if this specific precedent is the same as the main case
#             if current_name_fingerprint and current_name_fingerprint in normalize_legal_name(p):
#                 continue
#             filtered_precedents.append(p)
            
#         record["precedents"] = filtered_precedents
#         all_records.append(record)

#     # 4. Save to Parquet
#     df = pd.DataFrame(all_records)
#     df.to_parquet(output_path, engine='pyarrow', index=False, compression='snappy')

In [22]:
def build_final_parquet(text_folder, metadata_root, output_path):


    all_records = []
    text_files = [f for f in os.listdir(text_folder) if f.endswith('.txt')]

    for filename in tqdm(text_files, desc="Processing Cases"):
        with open(os.path.join(text_folder, filename), 'r', encoding='utf-8') as f:
            content = f.read()

        base_id = filename.replace('_EN.txt', '').replace('.txt', '')
        year = base_id.split('_')[0]
        json_path = os.path.join(metadata_root, year, "metadata", f"{base_id}.json")

        # Initialize full record
        record = {
            "file_id": base_id, "year": year, "full_text": content,
            "case_name": None, "precedents": [], "coram": None,
            "decision_date": None, "case_no": None, "disposal_nature": None,
            "neutral_citation": None
        }

        # 1. Load Metadata First
        if os.path.exists(json_path):
            with open(json_path, 'r', encoding='utf-8') as jf:
                meta_json = json.load(jf)
                record.update(extract_from_html(meta_json.get("raw_html", ""))) # Uses your updated HTML parser
                record["neutral_citation"] = meta_json.get("nc_display")

        # 2. Extract and Filter Precedents using the Fingerprint
        raw_precedents = extract_precedents(content)
        current_name_fingerprint = normalize_legal_name(record["case_name"])
        
        filtered_precedents = []
        for p in raw_precedents:
            p_fingerprint = normalize_legal_name(p)
            
            # Check if current case is inside the precedent or vice versa
            if current_name_fingerprint and (current_name_fingerprint in p_fingerprint or p_fingerprint in current_name_fingerprint):
                continue
            
            filtered_precedents.append(p)
            
        record["precedents"] = filtered_precedents
        all_records.append(record)

    # 3. Final Export
    df = pd.DataFrame(all_records)
    df.to_parquet(output_path, engine='pyarrow', index=False, compression='snappy')
    print(f"\nSaved {len(df)} cases to {output_path}")

In [23]:
start_year = str(input("Enter start year: "))
end_year = str(input("Enter end year: "))

if __name__ == "__main__":
    TEXT_DIR = f"../../My_Datasets/Text_Datasets/texts_{start_year}-{end_year}"
    META_ROOT = f"../../My_Datasets/SC_{start_year}-{end_year}"
    OUTPUT = f"../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_{start_year}-{end_year}.parquet"

    build_final_parquet(TEXT_DIR, META_ROOT, OUTPUT)

Processing Cases: 100%|██████████| 4368/4368 [00:12<00:00, 359.56it/s]



Saved 4368 cases to ../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_2020-2025.parquet


In [24]:
import pandas as pd

# Load the parquet file
df = pd.read_parquet(f"../../My_Datasets/Parquet_Datasets/SC_Parquet_Dataset_{start_year}-{end_year}.parquet")

# View the first 5 rows
print(df.head())

             file_id  year                                          full_text  \
0  2020_10_1043_1074  2020  M/S. L. R. BROTHERS INDO FLORA LTD.\nv.\nCOMMI...   
1  2020_10_1075_1131  2020  M/S BANDEKAR BROTHERS PVT. LTD. & ANR\nv.\nPRA...   
2  2020_10_1132_1150  2020  THE NEW INDIA ASSURANCE COMPANY LIMITED\nv.\nS...   
3    2020_10_135_237  2020  VINEETA SHARMA\nv.\nRAKESH SHARMA & ORS.\n(Civ...   
4       2020_10_1_26  2020  BCH ELECTRIC LIMITED\nv.\nPRADEEP MEHRA\n(Civi...   

                                           case_name  \
0  M/S. L. R. BROTHERS INDO FLORA LTD. versus COM...   
1  M/S BANDEKAR BROTHERS PVT. LTD. & ANR versus P...   
2  THE NEW INDIA ASSURANCE COMPANY LIMITED versus...   
3         VINEETA SHARMA versus RAKESH SHARMA & ORS.   
4          BCH ELECTRIC LIMITED versus PRADEEP MEHRA   

                                          precedents  \
0  [Commissioner of Central Excise and Customs v....   
1  [In Baban Singh and Anr. v. Jagdish Singh and ...   
2  [Magm

In [25]:
print(df.iloc[0].T)

file_id                                             2020_10_1043_1074
year                                                             2020
full_text           M/S. L. R. BROTHERS INDO FLORA LTD.\nv.\nCOMMI...
case_name           M/S. L. R. BROTHERS INDO FLORA LTD. versus COM...
precedents          [Commissioner of Central Excise and Customs v....
coram                             A.M. KHANWILKAR*, DINESH MAHESHWARI
decision_date                                              01-09-2020
case_no                                    CIVIL APPEAL No. 7157/2008
disposal_nature                                             Dismissed
neutral_citation                                          2020INSC525
Name: 0, dtype: object


In [30]:
filename = df.loc[3, "file_id"]
print(filename)

caseName = df.loc[3, 'case_name']
print(caseName)

2020_10_135_237
VINEETA SHARMA versus RAKESH SHARMA & ORS.


In [29]:
value = df.loc[3, 'precedents']
print(value)

['It was observed that the controversy concerning the interpretation of section 6 now stands settled with authoritative pronouncement in Prakash v. Phulavati which affirmed the view taken by the High Court as well as a Full Bench in Badrinarayan Shankar Bhandari v. Omprakash Shankar Bhandari'
 'Hindu law by all the coparceners. In KantaGoel v. B.P. Pathak'
 'Hardeo Rai v. Sakuntala Devi & Ors.'
 'Bill mentioned that nothing contained in the amended Section 6 should apply VINEETA SHARMA v. RAKESH SHARMA'
 'G. Sekar v. Geetha & Ors.'
 'AIR 1960 SC 335 and Mudigowda Gowdappa Sankh & Ors. v. Ramchandra Revgowda Sankh'
 'Bireswar Mookerji & Ors. v. Shib Chunder Roy'
 'G. Narasimulu & Ors. v. P. Basava Sankaram & Ors.'
 'Lord Westbury in Approvier v. Rama Subha Aiyan'
 'Rukhmabai v. Laxminarayan' 'In Kalwa Devdattam v. Union of India'
 'In Tek Bahadur Bhujil v. Debi Singh Bhujil'
 'In Baijnath Prasad Singh & Ors. v. Tej Bali Singh'
 'Section 6 of the Act deals with devolution of interest of 